In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
from xgboost import XGBClassifier

In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 1.6 MB/s eta 0:01:04
   ---------------------------------------- 1.0/101.7 MB 1.7 MB/s eta 0:00:58
    --------------------------------------- 1.8/101.7 MB 2.1 MB/s eta 0:00:47
    --------------------------------------- 2.4/101.7 MB 2.4 MB/s eta 0:00:43
   - -------------------------------------- 3.4/101.7 MB 2.8 MB/s eta 0:00:36
   - -------------------------------------- 4.5/101.7 MB 3.2 MB/s eta 0:00:31
   -- ------------------------------------- 6.0/101.7 MB 3.7 MB/s eta 0:00:26
   -- ------------------------------------- 7.6/101.7 MB 4.2 MB/s eta 0:00:23
   --- ------------------------------------ 8.9/101.7 MB 4.4 MB/s eta 0:00:21
   --- ------------------------------------ 9.4/101.7 MB 4.3 MB/s eta 0:00:22
   --- 


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: NVIDIA RTX A2000 Laptop GPU


In [7]:
df = pd.read_csv("../data/train.csv")

X = df.drop(columns=['target', 'ID_code'])
y = df['target']

# Feature Engineering
X['mean'] = X.mean(axis=1)
X['std'] = X.std(axis=1)
X['min'] = X.min(axis=1)
X['max'] = X.max(axis=1)
X['range'] = X['max'] - X['min']

X.shape

C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_21684\2715614683.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['mean'] = X.mean(axis=1)
C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_21684\2715614683.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['std'] = X.std(axis=1)
C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_21684\2715614683.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joinin

(200000, 205)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((160000, 205), (40000, 205))

In [9]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=9,
    random_state=42,
    eval_metric='logloss',
    tree_method='hist',
    device='cuda'   # 🔥 GPU
)

In [10]:
# Train
model.fit(X_train, y_train)

# Predict probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Apply best threshold
y_pred = (y_prob >= 0.6).astype(int)

# Evaluation
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

F1: 0.5092570892898992
ROC-AUC: 0.8700315819457465


d:\6th semester study\Regrerssion-Comparison\research\Lib\site-packages\xgboost\core.py:751: UserWarning: [19:29:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


## 🚀 Advanced Model: XGBoost (GPU)

### Objective
To improve classification performance using a powerful gradient boosting model with GPU acceleration.

---

### Approach

- Used XGBoost classifier with GPU support
- Applied feature engineering from previous experiments
- Handled class imbalance using `scale_pos_weight`
- Applied optimal threshold (0.6)

---

### Results

| Model | F1-score | ROC-AUC |
|------|----------|--------|
| Logistic Regression + FE | ~0.47 | ~0.864 |
| XGBoost (GPU) | **~0.51** | **~0.87** |

---

### Key Observations

- XGBoost outperformed Logistic Regression
- Significant improvement in F1-score
- ROC-AUC also improved slightly
- Model better captured complex patterns in data

---

### Insight

- Tree-based boosting models handle high-dimensional tabular data effectively
- GPU acceleration significantly reduces training time
- Combining feature engineering with strong models yields best results

---

### Conclusion

- XGBoost is a strong candidate for final model
- Further improvements can be achieved via hyperparameter tuning